# Global Superstore 2016 — Sales & Profit Analysis

**Student:** Sachin Sisodiya  
**Project Type:** Exploratory Data Analysis and Business Insights  
**Dataset:** Global Superstore 2016

This notebook loads the provided Excel dataset, cleans the data, performs exploratory and statistical analysis, answers business questions, creates visualizations, and summarizes actionable insights.


## 1. Project Objectives

- Understand sales, profit, quantity and order performance.
- Analyze trends over time.
- Compare markets, regions, categories, sub-categories and customer segments.
- Identify high-performing and loss-making products.
- Examine shipping performance.
- Analyze returned orders.
- Explore the relationship between discount and profit.


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")


In [ ]:
# Load the dataset
# Put global_superstore_2016.xlsx in the same folder as this notebook.
file_path = "global_superstore_2016.xlsx"

orders = pd.read_excel(file_path, sheet_name="Orders")
returns = pd.read_excel(file_path, sheet_name="Returns")
people = pd.read_excel(file_path, sheet_name="People")

print("Orders:", orders.shape)
print("Returns:", returns.shape)
print("People:", people.shape)


In [ ]:
# Inspect the data
display(orders.head())
print("\nColumns:")
print(orders.columns.tolist())

print("\nData types:")
display(orders.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(orders.isna().sum().sort_values(ascending=False).to_frame("missing"))


## 2. Data Cleaning and Feature Engineering

The dataset contains three sheets: `Orders`, `Returns`, and `People`.

The `Postal Code` field has many missing values, but it is not required for the core sales/profit analysis. Date fields are converted to datetime, and additional fields are created for year, shipping days, profit margin, and returned-order analysis.


In [ ]:
# Convert dates
orders["Order Date"] = pd.to_datetime(orders["Order Date"])
orders["Ship Date"] = pd.to_datetime(orders["Ship Date"])

# Feature engineering
orders["Year"] = orders["Order Date"].dt.year
orders["Month"] = orders["Order Date"].dt.month
orders["Ship Days"] = (orders["Ship Date"] - orders["Order Date"]).dt.days
orders["Profit Margin %"] = np.where(
    orders["Sales"] != 0,
    orders["Profit"] / orders["Sales"] * 100,
    0
)

# Map returned orders
returned_ids = set(
    returns.loc[returns["Returned"].astype(str).str.lower().eq("yes"), "Order ID"]
)
orders["Returned"] = orders["Order ID"].isin(returned_ids)

print("Cleaning and feature engineering completed.")


In [ ]:
# Core KPIs
kpis = pd.Series({
    "Total Sales": orders["Sales"].sum(),
    "Total Profit": orders["Profit"].sum(),
    "Total Quantity": orders["Quantity"].sum(),
    "Unique Orders": orders["Order ID"].nunique(),
    "Unique Customers": orders["Customer ID"].nunique(),
    "Unique Products": orders["Product ID"].nunique(),
    "Profit Margin %": orders["Profit"].sum() / orders["Sales"].sum() * 100
})
display(kpis.to_frame("Value"))


## 3. Business Question 1 — How did sales and profit change by year?

In [ ]:
yearly = orders.groupby("Year").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum"),
    Orders=("Order ID", "nunique")
)
yearly["Profit Margin %"] = yearly["Profit"] / yearly["Sales"] * 100
display(yearly.round(2))

yearly[["Sales", "Profit"]].plot(marker="o", figsize=(9,5))
plt.title("Sales and Profit Trend by Year")
plt.xlabel("Year")
plt.ylabel("Amount")
plt.show()


## 4. Business Question 2 — Which product categories generate the most sales and profit?

In [ ]:
category = orders.groupby("Category").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum"),
    Orders=("Order ID", "nunique")
).sort_values("Sales", ascending=False)

display(category.round(2))

category[["Sales", "Profit"]].plot(kind="bar", figsize=(9,5))
plt.title("Sales and Profit by Category")
plt.xlabel("Category")
plt.ylabel("Amount")
plt.xticks(rotation=0)
plt.show()


## 5. Business Question 3 — Which markets contribute most to sales and profit?

In [ ]:
market = orders.groupby("Market").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
).sort_values("Sales", ascending=False)

display(market.round(2))

market["Profit"].sort_values().plot(kind="barh", figsize=(9,5))
plt.title("Profit by Market")
plt.xlabel("Profit")
plt.show()


## 6. Business Question 4 — Which customer segments generate the most sales?

In [ ]:
segment = orders.groupby("Segment").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
).sort_values("Sales", ascending=False)

display(segment.round(2))

segment["Sales"].sort_values().plot(kind="barh", figsize=(9,5))
plt.title("Sales by Customer Segment")
plt.xlabel("Sales")
plt.show()


## 7. Business Question 5 — What are the best and worst-performing products?

In [ ]:
product_summary = orders.groupby("Product Name").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Quantity=("Quantity", "sum")
)

top_products = product_summary.sort_values("Sales", ascending=False).head(10)
loss_products = product_summary.sort_values("Profit").head(10)

print("Top 10 products by sales")
display(top_products.round(2))

print("10 products with the lowest profit")
display(loss_products.round(2))


## 8. Business Question 6 — Which countries have the highest sales?

In [ ]:
top_countries = orders.groupby("Country").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Orders=("Order ID", "nunique")
).sort_values("Sales", ascending=False).head(10)

display(top_countries.round(2))


## 9. Business Question 7 — How does shipping mode affect order volume and delivery time?

In [ ]:
ship = orders.groupby("Ship Mode").agg(
    Sales=("Sales", "sum"),
    Profit=("Profit", "sum"),
    Avg_Ship_Days=("Ship Days", "mean"),
    Orders=("Order ID", "nunique")
).sort_values("Orders", ascending=False)

display(ship.round(2))


## 10. Business Question 8 — What is the return rate?

In [ ]:
total_orders = orders["Order ID"].nunique()
returned_orders = orders.loc[orders["Returned"], "Order ID"].nunique()
return_rate = returned_orders / total_orders * 100

print(f"Returned orders: {returned_orders:,}")
print(f"Total orders: {total_orders:,}")
print(f"Return rate: {return_rate:.2f}%")


## 11. Statistical Analysis — Discount vs Profit

Pearson correlation is used to measure the linear relationship between discount and profit. Correlation does not prove causation; it only indicates the strength and direction of a linear association.


In [ ]:
discount_profit_corr = orders["Discount"].corr(orders["Profit"])
print(f"Pearson correlation between Discount and Profit: {discount_profit_corr:.4f}")

plt.figure(figsize=(8,5))
sns.scatterplot(data=orders.sample(min(10000, len(orders)), random_state=42),
                x="Discount", y="Profit", alpha=0.35)
plt.title("Discount vs Profit")
plt.show()


## 12. Key Findings

1. Sales and profit increased substantially from 2012 to 2015.
2. Technology generated the highest category sales in this dataset.
3. Asia Pacific generated the highest market sales, while Europe generated the highest market profit.
4. Consumer was the largest customer segment by sales.
5. Standard Class was the most frequently used shipping mode.
6. A small group of products generated significant losses and should be reviewed for pricing, discounting, cost or demand issues.
7. The discount-profit correlation is negative in this dataset; this is an association, not proof that discounts alone caused lower profit.
8. Returned orders represent a measurable share of total orders and can be monitored as a quality/service KPI.


## 13. Conclusion

The analysis shows how Global Superstore data can be used to monitor revenue, profitability, customers, products, markets, shipping and returns. The resulting KPIs and visualizations can support decisions around product portfolio management, regional performance, pricing/discount strategy and operational improvement.
